# gigapath_runpod (로컬→RunPod SSH 오케스트레이션)

로컬 주피터 노트북에서 SSH/rsync로 RunPod 인스턴스를 제어하며 **SVS 업로드 → 타일링 → 원본 삭제 → 벡터화 → 타일 삭제 → 학습** 순서로 스토리지 사용을 최소화합니다. 아래 자리표시자(타일링/벡터화/학습 커맨드, 호스트명 등)를 환경에 맞게 수정하세요.


## 0. 기본 설정 (호스트/경로/커맨드 자리표시자)

- `SSH_HOST_DIRECT/PORT`: RunPod Direct TCP (SCP/rsync 가능)
- `SSH_HOST_GATEWAY`: 필요 시 게이트웨이 사용 (SCP 제한)
- `REMOTE_RAW`, `REMOTE_WORK`: RunPod 내부 경로
- 타일링/벡터화/학습 커맨드를 gigapath 코드에 맞게 채우기
- 삭제 정책 토글: `DELETE_SVS_AFTER_TILING`, `DELETE_TILES_AFTER_FEATURE`


In [1]:
import subprocess, shlex, time
from pathlib import Path

# SSH 대상 및 키 (실제 RunPod 접속 정보로 설정)
# 게이트웨이(ssh.runpod.io)와 Direct TCP(root@IP -p PORT)를 모두 지원
SSH_HOST_GATEWAY = "s2idinmwnlob85-64410a8c@ssh.runpod.io"  # proxied, SCP 제한
SSH_HOST_DIRECT = "root@64.247.206.204"  # direct TCP, SCP/rsync 가능
SSH_PORT_DIRECT = 44922
SSH_KEY = "~/.ssh/runpod_peter"  # 필요 없으면 None
# 기본은 Direct TCP를 사용(PTY 오류 회피, rsync/ssh 모두 동일 경로)
USE_DIRECT_FOR_SSH = True
USE_DIRECT_FOR_RSYNC = True
# PTY 옵션: direct에서는 비워두고, 게이트웨이 필요 시 설정 (예: "-T")
SSH_EXTRA_OPTS = ""

# RunPod 내부 경로
REMOTE_RAW = "~/data/raw"
REMOTE_WORK = "~/data/work"
REMOTE_CHECKPOINT = f"{REMOTE_WORK}/checkpoints"
REMOTE_LOG = "~/logs"

# 타일링/벡터화/학습 커맨드 템플릿 (gigapath 코드에 맞게 수정)
TILE_SIZE = 224
TILE_OVERLAP = 0
TILING_CMD_TEMPLATE = (
    "python tools/tile_svs.py --input {input} --output {output} "
    "--tile-size {tile_size} --overlap {overlap}"
)
FEATURE_CMD_TEMPLATE = "python tools/extract_features.py --tiles {tiles} --out {out}"
TRAIN_CMD_TEMPLATE = (
    "python train.py --features {features_dir} --out {ckpt_dir} --logdir {log_dir}"
)

# 삭제 정책
DELETE_SVS_AFTER_TILING = True
DELETE_TILES_AFTER_FEATURE = True

print("SSH_HOST_DIRECT=", SSH_HOST_DIRECT, "port", SSH_PORT_DIRECT)
print("SSH_HOST_GATEWAY=", SSH_HOST_GATEWAY)
print("REMOTE_RAW=", REMOTE_RAW)
print("REMOTE_WORK=", REMOTE_WORK)
print("TILING_CMD_TEMPLATE=", TILING_CMD_TEMPLATE)
print("FEATURE_CMD_TEMPLATE=", FEATURE_CMD_TEMPLATE)
print("TRAIN_CMD_TEMPLATE=", TRAIN_CMD_TEMPLATE)


SSH_HOST_DIRECT= root@64.247.206.204 port 44922
SSH_HOST_GATEWAY= s2idinmwnlob85-64410a8c@ssh.runpod.io
REMOTE_RAW= ~/data/raw
REMOTE_WORK= ~/data/work
TILING_CMD_TEMPLATE= python tools/tile_svs.py --input {input} --output {output} --tile-size {tile_size} --overlap {overlap}
FEATURE_CMD_TEMPLATE= python tools/extract_features.py --tiles {tiles} --out {out}
TRAIN_CMD_TEMPLATE= python train.py --features {features_dir} --out {ckpt_dir} --logdir {log_dir}


## 1. 유틸리티: 로컬 실행/SSH/rsync 래퍼

- `run_local`: 로컬 셸 실행
- `run_ssh`: RunPod에서 명령 실행
- `rsync_upload` / `rsync_download`: 부분 업로드/다운로드 지원


In [2]:
def run_local(cmd, check=True):
    print(f"[local] $ {cmd}")
    result = subprocess.run(cmd, shell=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"Local command failed: {cmd}")
    return result.returncode

def _ssh_parts(host, port=None):
    parts = ["ssh"]
    if SSH_KEY:
        parts += ["-i", SSH_KEY]
    if SSH_EXTRA_OPTS:
        parts += shlex.split(SSH_EXTRA_OPTS)
    if port:
        parts += ["-p", str(port)]
    parts += [host]
    return parts

def run_ssh(cmd, check=True, use_direct=None):
    if use_direct is None:
        use_direct = USE_DIRECT_FOR_SSH
    parts = _ssh_parts(
        SSH_HOST_DIRECT if use_direct else SSH_HOST_GATEWAY,
        SSH_PORT_DIRECT if use_direct else None,
    )
    ssh_cmd = " ".join(shlex.quote(p) for p in parts + [cmd])
    print(f"[ssh] $ {cmd}")
    result = subprocess.run(ssh_cmd, shell=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"SSH command failed: {cmd}")
    return result.returncode

def rsync_upload(local_path: Path, remote_dir: str):
    parts = _ssh_parts(
        SSH_HOST_DIRECT if USE_DIRECT_FOR_RSYNC else SSH_HOST_GATEWAY,
        SSH_PORT_DIRECT if USE_DIRECT_FOR_RSYNC else None,
    )
    ssh_opt = " ".join(shlex.quote(p) for p in parts)
    cmd = (
        f"rsync -av --partial --progress -e {shlex.quote(ssh_opt)} "
        f"{shlex.quote(str(local_path))} {SSH_HOST_DIRECT if USE_DIRECT_FOR_RSYNC else SSH_HOST_GATEWAY}:{shlex.quote(remote_dir)}/"
    )
    run_local(cmd)

def rsync_download(remote_path: str, local_dir: Path):
    parts = _ssh_parts(
        SSH_HOST_DIRECT if USE_DIRECT_FOR_RSYNC else SSH_HOST_GATEWAY,
        SSH_PORT_DIRECT if USE_DIRECT_FOR_RSYNC else None,
    )
    ssh_opt = " ".join(shlex.quote(p) for p in parts)
    local_dir.mkdir(parents=True, exist_ok=True)
    cmd = (
        f"rsync -av --partial --progress -e {shlex.quote(ssh_opt)} "
        f"{SSH_HOST_DIRECT if USE_DIRECT_FOR_RSYNC else SSH_HOST_GATEWAY}:{shlex.quote(remote_path)} {shlex.quote(str(local_dir))}/"
    )
    run_local(cmd)


## 2. 원격 기본 디렉토리 준비 및 점검

- 필요한 폴더 생성
- GPU/디스크 확인 (실패 시에도 진행되도록 `|| true`)
- `ssh` 접속 테스트까지 포함


In [3]:
# 접속 및 경로 생성
run_ssh(f"mkdir -p {REMOTE_RAW} {REMOTE_WORK} {REMOTE_CHECKPOINT} {REMOTE_LOG}")
run_ssh("echo 'SSH OK on $(hostname)' && nvidia-smi || true", check=False)
run_ssh("df -h . || true", check=False)


[ssh] $ mkdir -p ~/data/raw ~/data/work ~/data/work/checkpoints ~/logs


Host key verification failed.


RuntimeError: SSH command failed: mkdir -p ~/data/raw ~/data/work ~/data/work/checkpoints ~/logs

## 3. SVS 업로드 (로컬 → RunPod)

- 로컬 파일을 하나 선택해 업로드
- `rsync`는 부분 재시작을 지원


In [ ]:
# 예시: 로컬 SVS 경로 지정 후 업로드
local_svs = Path("/path/to/local/file.svs")  # TODO: 실제 경로로 수정
if local_svs.exists():
    rsync_upload(local_svs, REMOTE_RAW)
else:
    print("로컬 SVS 경로를 설정하세요: local_svs = Path('...file.svs')")


## 4. 원격 타일링 함수 (SVS 1개)

- RunPod에서 타일링 후 옵션에 따라 원본 삭제
- `TILING_CMD_TEMPLATE`를 gigapath에 맞게 수정


In [ ]:
def remote_tile(svs_remote_path: str) -> str:
    svs_name = Path(svs_remote_path).name
    tile_dir = f"{REMOTE_WORK}/{Path(svs_name).stem}_tiles"
    cmd = TILING_CMD_TEMPLATE.format(
        input=shlex.quote(svs_remote_path),
        output=shlex.quote(tile_dir),
        tile_size=TILE_SIZE,
        overlap=TILE_OVERLAP,
    )
    run_ssh(cmd)
    if DELETE_SVS_AFTER_TILING:
        run_ssh(f"rm -f {shlex.quote(svs_remote_path)}")
        print(f"Deleted original svs: {svs_remote_path}")
    return tile_dir


## 5. 원격 벡터화 함수 (타일 디렉토리 1개)

- 타일 디렉토리를 받아 feature 파일 생성
- 완료 후 옵션에 따라 타일 삭제
- `FEATURE_CMD_TEMPLATE`를 gigapath에 맞게 수정


In [ ]:
def remote_featurize(tile_dir: str) -> str:
    feat_path = f"{tile_dir}_feats.npy"
    cmd = FEATURE_CMD_TEMPLATE.format(
        tiles=shlex.quote(tile_dir),
        out=shlex.quote(feat_path),
    )
    run_ssh(cmd)
    if DELETE_TILES_AFTER_FEATURE:
        run_ssh(f"rm -rf {shlex.quote(tile_dir)}")
        print(f"Deleted tiles: {tile_dir}")
    print(f"Feature saved: {feat_path}")
    return feat_path


## 6. 큐 처리: RAW에 있는 SVS를 하나씩 순차 처리

- 이름순으로 n개(`limit`) 처리
- 타일링→원본 삭제→벡터화→타일 삭제


In [ ]:
def list_remote_svs():
    cmd = f"ls -1 {REMOTE_RAW}/*.svs 2>/dev/null"
    parts = _ssh_parts(
        SSH_HOST_DIRECT if USE_DIRECT_FOR_SSH else SSH_HOST_GATEWAY,
        SSH_PORT_DIRECT if USE_DIRECT_FOR_SSH else None,
    )
    ssh_cmd = " ".join(shlex.quote(p) for p in parts + [cmd])
    output = subprocess.run(ssh_cmd, shell=True, capture_output=True, text=True)
    if output.returncode != 0:
        return []
    return [line.strip() for line in output.stdout.splitlines() if line.strip()]

def process_queue(limit: int = 1):
    svs_list = list_remote_svs()
    if not svs_list:
        print("REMOTE_RAW에 svs가 없습니다. rsync로 업로드하세요.")
        return []
    feats = []
    for svs_path in svs_list[:limit]:
        start = time.time()
        print(f"=== Start {svs_path} ===")
        tile_dir = remote_tile(svs_path)
        feat_path = remote_featurize(tile_dir)
        elapsed = time.time() - start
        print(f"=== Done {svs_path} | feature: {feat_path} | {elapsed/60:.1f} min ===")
        feats.append(feat_path)
    return feats


## 7. 원격 학습 실행 (옵션)

- `RUN_TRAINING=True`로 토글
- `TRAIN_CMD_TEMPLATE`를 gigapath 학습 스크립트에 맞게 수정
- 체크포인트/로그 경로는 REMOTE_CHECKPOINT/REMOTE_LOG


In [ ]:
RUN_TRAINING = False  # True로 바꾸면 학습 실행

def run_remote_training():
    train_cmd = TRAIN_CMD_TEMPLATE.format(
        features_dir=shlex.quote(REMOTE_WORK),
        ckpt_dir=shlex.quote(REMOTE_CHECKPOINT),
        log_dir=shlex.quote(REMOTE_LOG),
    )
    print(f"Training command:
{train_cmd}
")
    run_ssh(train_cmd)

if RUN_TRAINING:
    run_remote_training()
else:
    print("Training skipped (set RUN_TRAINING=True to run)")


## 8. 결과 다운로드/아카이브

- 필요 체크포인트/feature를 로컬로 rsync 다운로드
- 오래된 체크포인트는 RunPod에서 압축 후 삭제


In [ ]:
def download_checkpoints(local_dir: Path):
    rsync_download(f"{REMOTE_CHECKPOINT}/", local_dir)

def remote_archive_and_delete(path_glob: str):
    # 예: path_glob="~/data/work/checkpoints/epoch*"
    cmd = f"for p in {path_glob}; do [ -e "$p" ] || continue; tar -czf ${p}.tar.gz -C $(dirname $p) $(basename $p) && rm -rf $p; done"
    run_ssh(cmd)


## 9. 추천 워크플로우 (셀 실행 순서)

1) 0~2 셀 실행: 설정/접속/경로 준비
2) 3 셀에서 로컬 SVS 경로 지정 후 업로드 (또는 터미널에서 직접 rsync)
3) 6 셀 `process_queue(limit=1)` 실행: 타일링→벡터화→삭제 순차 처리
4) 필요시 7 셀에서 학습 실행 (`RUN_TRAINING=True`)
5) 8 셀로 체크포인트/결과를 로컬로 rsync 다운로드

공간을 아끼려면 항상: 타일링 후 원본 삭제, 벡터화 후 타일 삭제, 체크포인트는 주기적으로 압축/정리.
